# Lab 03: Loops, Telemetry, and Evidence
Introduction to Programming — Space Engineering, 2026/2027  
Provisional version · Name: *Mouizeddine Boulkroun* · Date: *11/26/2026*

In Lab 02 you made a decision for one spacecraft reading. Now you will apply rules to a sequence of readings and produce a useful summary.

By the end, aim to:
- combine `for` loops with `if/elif/else` and Boolean conditions;
- distinguish the current reading from information accumulated across readings;
- use `range()` without losing an endpoint;
- diagnose code that runs but gives the wrong result;
- write a precise specification and test an LLM's interpretation of it.

## Route

| Portion | Activities | Planning time |
|---|---|---:|
| Part A: in class | A1–A5: trace, modify, repair, build | 75 minutes |
| Class checkpoint | Paper exit exercise, clean run, save and commit | 15 minutes |
| Part B: at home | B1–B4: batch monitor, tests, prompt audit | 60–90 minutes |

Times are estimates. If you finish an activity early, use its stretch question rather than waiting. The optional extension is not required homework.

Try the short class traces and your first diagnosis yourself before using an LLM. At home, assistance is allowed: record what you asked, what you changed, and how you checked it. A detailed prompt helps communicate your intent; it does not guarantee correct code.

Use your existing course environment. This notebook needs only standard Python, with no downloads or additional packages. All telemetry here is synthetic and the rules are simplified teaching rules.

For each repair: identify a failing input → state expected and actual behavior → explain the cause → change the code → rerun the test. Repair the marked cells in place; the original faulty code remains in the supplied notebook and your Git history.

## Part A — in class
### A1. One loop, several decisions · about 12 minutes

A list holds an ordered sequence: `[20, 76, 90]`. A `for` loop assigns each value in turn to its loop variable and runs the indented block for that value. This is one *iteration*. The loop visits values in list order, including repeated values.

Here the outer indentation belongs to the loop; the deeper indentation belongs to a conditional inside that loop. The final `print()` is outside the loop.

Use these temperature rules in A1 and A2:
1. `OUTSIDE LIMITS` below −20°C or above 85°C.
2. Otherwise, `WARM` above 75°C.
3. Otherwise, `ACCEPTED`.

Trace just the first three readings by hand, then predict the final line. Run the cell once. Explain why 85°C and 86°C take different branches.

| Reading, °C | Predicted status | Observed status |
|---:|---|---|
| 20 | ACCEPTED| ACCEPTED |
| 85 | WARM | WARM |
| 86 | OUTSIDE LIMITS | OUTSIDE LIMITS |

Predicted final line:  
Why the boundary matters:

In [1]:
temperatures_c = [20, 85, 86, -20, -21, 75, 76]

for temperature_c in temperatures_c:
    if temperature_c < -20 or temperature_c > 85:
        status = "OUTSIDE LIMITS"
    elif temperature_c > 75:
        status = "WARM"
    else:
        status = "ACCEPTED"
    print(temperature_c, "C:", status)

print("Assessment complete")

20 C: ACCEPTED
85 C: WARM
86 C: OUTSIDE LIMITS
-20 C: ACCEPTED
-21 C: OUTSIDE LIMITS
75 C: ACCEPTED
76 C: WARM
Assessment complete


### Make a change that tests the structure

In the cell above:
1. Add a repeated value of 86 at the end of the list. Predict how many output lines it adds.
2. Move the final message inside the loop, run it, and explain the difference. Then restore it outside.
3. Restore the original list before continuing.

Discuss: if you moved `print(temperature_c, "C:", status)` outside the loop, what information would be lost?

Stretch: reorder the conditions so `temperature_c > 75` comes first. Which input exposes the problem? Restore the correct order afterward.

### A2. Remembering a count and a total · about 18 minutes

A counter starts before the loop and increases when an event occurs. `count = count + 1` reads the old value, adds one, and stores the result. A total uses the same pattern but adds a measurement instead of one.

Requirement: count every reading outside the inclusive interval −20°C to 85°C. Report one final count, including zero when no readings are outside limits.

The following code runs but is wrong. Before repairing it, trace `outside_count` after each iteration for `[90, 20, 91, 30]`. Find the line that prevents it remembering earlier readings.

In [2]:
temperatures_c = [90, 20, 91, 30]
outside_count = 0

outside_count = 0
for temperature_c in temperatures_c:
    if temperature_c < -20 or temperature_c > 85:
        outside_count = outside_count + 1

print("Readings outside limits:", outside_count)

Readings outside limits: 2


My trace of the faulty count:  `0` in all the iterations <br>
Required final count:  `2` <br>
Actual faulty final count:  `0` <br>
Cause and repair: Putting `outside_count = 0` inside the loop resets the counter on every iteration. Repair it by moving `outside_count = 0` outside (above) the for loop. <br>

After repairing, also test `[20, 30]`, `[90, 91]`, and `[]`. The last is an empty list: the loop body runs zero times. Record the results, then restore the original input.

| Input | Expected count | Observed count |
|---|---:|---:|
| `[90, 20, 91, 30]` | 2 |2 |
| `[20, 30]` | 0|0 |
| `[90, 91]` | 2| 2|
| `[]` | 0|0 |

### Build a filtered summary

Now calculate the mean of readings inside the inclusive interval −20°C to 85°C. WARM readings are still inside this interval and must be included. Exclude every outside reading from both the count and total.

Plan where to put:
- initialization of a count and total;
- the condition deciding whether a reading contributes;
- the final division and output.

Use a loop, a count, and a total; do not use `sum()` for this exercise. If there are no included readings, print `No readings inside limits` instead of dividing by zero. `[]` is also a valid input.

For `[20, 86, -20, 80]`, the included count is 3, the total is 80°C, and the mean is approximately 26.67°C. A count is a number of readings, not a temperature.

In [3]:
temperatures_c = [20, 86, -20, 80]

inside_count = 0
total_temp = 0

for temperature_c in temperatures_c:
    if -20 <= temperature_c <= 85:
        inside_count = inside_count + 1
        total_temp = total_temp + temperature_c
        
if inside_count > 0:
    mean_temp = total_temp / inside_count
    print("Included count:", inside_count)
    print("Mean temperature:", mean_temp)
else:
    print("No readings inside limits")

Included count: 3
Mean temperature: 26.666666666666668


Check with `[86, -21]` and `[]` as well. Restore the original input afterward.

Why would dividing by the length of the original input list give the wrong mean?  
    => Because the original list length includes readings that were excluded (like 86), which would act as the wrong denominator and artificially skew/lower the calculated average. <br>

What happens if you move initialization inside the loop?
    => `inside_count` and `total_temp` would reset to 0 at the start of every single iteration, preventing accumulation and breaking the total and count logic entirely.

### A3. Repeating a calculation with `range()` · about 12 minutes

`range(start, stop)` supplies integers starting at `start` and stopping *before* `stop`. For example, `range(2, 5)` supplies 2, 3, 4. `range(4)` supplies 0, 1, 2, 3.

Requirement: begin at 50% charge. Simulate exactly five time steps, subtracting 7 percentage points at each step. Print steps 1 through 5 with the charge **after** that step's loss. These are fixed losses, not a 7% proportional decrease.

The code below runs, but it violates the requirement in two ways. Predict the final printed step and charge before fixing it. The required last line must describe step 5 at 15%.

In [4]:
battery_charge = 50
loss_per_step = 7

for step in range(1, 6):
    battery_charge = battery_charge - loss_per_step
    
    if battery_charge < 15:
        status = "CRITICAL"
    elif battery_charge < 25:
        status = "SAFE MODE"
    elif battery_charge <= 35:
        status = "WARNING"
    else:
        status = "NOMINAL"
        
    print("Step", step, "charge:", battery_charge, "% - Status:", status) # if this was above the substraction logic, it should be "range(1, 7)"

Step 1 charge: 43 % - Status: NOMINAL
Step 2 charge: 36 % - Status: NOMINAL
Step 3 charge: 29 % - Status: WARNING
Step 4 charge: 22 % - Status: SAFE MODE
Step 5 charge: 15 % - Status: SAFE MODE


1. Faulty last printed step and charge:  Step 4 charge: 29 % <br>
2. Repair 1 and why it is needed:  Change `range(1, 5)`  to `range(1, 6)`. Because `range()` stops before the second argument, 5 is excluded, so it only runs 4 times instead of the required 5.<br>
3. Repair 2 and why it is needed: Reorder the statements inside the loop so that you subtract the loss before printing, or print the updated charge after the subtraction. The requirement states to print the charge after that step's loss.<br>

After fixing the simulation, add an `if/elif/else` chain inside the loop to print the battery status after each loss:
- CRITICAL below 15%;
- SAFE MODE below 25%;
- WARNING at most 35%;
- NOMINAL otherwise.

Use the newly calculated charge, not the previous step's charge. At 15%, which status is required?
-> `WARNING` (since the rules say: `CRITICAL below 15%`, SAFE MODE below 25%, WARNING at most 35%—so 15% falls under WARNING).

Stretch: change the simulation to six steps. Explain why this is a simulation of evolving state, whereas A1 classified independent readings. Restore five steps afterward.

### A4. Plausible code, misleading result · about 13 minutes

Requirement: assess each battery reading independently using the same four battery rules from A3. Print exactly one status per reading. A previous reading must not affect the next status.

This code runs, and every comparison looks plausible. Trace the assignments to `status` for the reading 10 before running it. Then repair the code in place.

In [5]:
# battery_readings = [10, 30, 80]

# for battery_charge in battery_readings:
#     status = "NOMINAL"
#     if battery_charge < 15:
#         status = "CRITICAL"
#     if battery_charge < 25:
#         status = "SAFE MODE"
#     if battery_charge <= 35:
#         status = "WARNING"
#     else:
#         status = "NOMINAL"
#     print(battery_charge, "%:", status)

battery_readings = [10, 30, 80]

for battery_charge in battery_readings:
    if battery_charge < 15:
        status = "CRITICAL"
    elif battery_charge < 25:
        status = "SAFE MODE"
    elif battery_charge <= 35:
        status = "WARNING"
    else:
        status = "NOMINAL"
    print(battery_charge, "%:", status)

10 %: CRITICAL
30 %: WARNING
80 %: NOMINAL


1. Assignments to `status` for 10%, in order:  `NOMINAL` -> `CRITICAL` -> `SAFE MODE` -> `WARNING` (because all three independent if statements run sequentially, overwriting each other, and 10 is less than 15, 25, and 35). <br>
2. Which `if` owns the `else` in the faulty code? The final else belongs exclusively to the `closest if` statement (`if battery_charge <= 35:`).
3. My repair and its reason: Combine the conditional branches into a proper mutually exclusive chain using `elif` instead of multiple separate `if` statements so that it stops checking once a condition is met.

### A different mistake: carrying a status forward

A proposed replacement uses one decision chain, but initializes `status` only once and omits the final `else`. Predict what happens for `[10, 80]`, then run it.

Repair it so that every reading receives its own status. Test both `[10, 80]` and `[80, 10]`: reversing the inputs should reverse the corresponding statuses, without changing any reading's classification.

In [6]:
# battery_readings = [10, 80]
# status = "NOMINAL"

# for battery_charge in battery_readings:
#     if battery_charge < 15:
#         status = "CRITICAL"
#     elif battery_charge < 25:
#         status = "SAFE MODE"
#     elif battery_charge <= 35:
#         status = "WARNING"
#     print(battery_charge, "%:", status)

battery_readings = [10, 80]

for battery_charge in battery_readings:
    status = "NOMINAL"

    if battery_charge < 15:
        status = "CRITICAL"
    elif battery_charge < 25:
        status = "SAFE MODE"
    elif battery_charge <= 35:
        status = "WARNING"
    print(battery_charge, "%:", status)

10 %: CRITICAL
80 %: NOMINAL


Expected and faulty observed statuses for `[10, 80]`:  
*   Expected: 10% is CRITICAL, 80% should be NOMINAL.
*   Faulty Observed: 10% is CRITICAL, but 80% incorrectly inherits CRITICAL as well because status was initialized outside the loop and never reset.

Why a single reading of `[80]` would hide the mistake:  Because if you only tested `[80]`, it would correctly `hit no conditions`, keep the default initialized state, and output `NOMINAL`, masking the bug that happens when moving between different states.

My repair: Move `status = "NOMINAL"` inside the loop so that it resets freshly for every new reading.

### A5. Build a battery report · about 20 minutes

Combine the ideas in a new cell. Use the four battery rules above. For each reading, print its value and one status. After the loop, print the number of readings in each category.

Requirements:
- Visit every reading, including repeated values.
- Each reading contributes to exactly one category.
- The four counts must add up to the number of input readings.
- An empty list produces four zero counts and no per-reading lines.
- Running the whole cell twice must produce the same output.

Before coding, complete the count predictions for the supplied data. Use ordinary variables for the four counters; dictionaries and functions are not needed. <br>

* 80: NOMINAL
* 35: WARNING
* 25: WARNING
* 15: WARNING
* 14: CRITICAL
* 36: NOMINAL
* 24: SAFE MODE
* 10: CRITICAL
* 80: NOMINAL

| CRITICAL | SAFE MODE | WARNING | NOMINAL | Total readings |
|---:|---:|---:|---:|---:|
|2 |1 |3 |3 |9 |

My plan in 3–5 steps:
1. Initialize four counters to `0` (one for each category).
2. Loop through each `battery_charge` in `battery_readings`.
3. Determine the status using `if/elif/else` and print the reading alongside its status.
4. Increment the corresponding counter for that category.
5. Print the final counts for all four categories after the loop ends.

In [7]:
battery_readings = [80, 35, 25, 15, 14, 36, 24, 10, 80]

# Initialize four counters.
critical_count = 0
safe_count = 0
warning_count = 0
nominal_count = 0

# For each reading: select one status, update its counter, and print.
for battery_charge in battery_readings:
    if battery_charge < 15:
        status = "CRITICAL"
        critical_count = critical_count + 1
    elif battery_charge < 25:
        status = "SAFE MODE"
        safe_count = safe_count + 1
    elif battery_charge <= 35:
        status = "WARNING"
        warning_count = warning_count + 1
    else:
        status = "NOMINAL"
        nominal_count = nominal_count + 1
        
    print(battery_charge, "%:", status)

# Print the four final counts outside the loop.
print("\nFinal Report:")
print("CRITICAL:", critical_count)
print("SAFE MODE:", safe_count)
print("WARNING:", warning_count)
print("NOMINAL:", nominal_count)

80 %: NOMINAL
35 %: WARNING
25 %: WARNING
15 %: SAFE MODE
14 %: CRITICAL
36 %: NOMINAL
24 %: SAFE MODE
10 %: CRITICAL
80 %: NOMINAL

Final Report:
CRITICAL: 2
SAFE MODE: 2
NOMINAL: 3


Test your report with `[10, 80]`, `[35, 35]`, and `[]`, then restore the supplied data. For each, predict the counts first.

| Input | Expected counts (critical, safe, warning, nominal) | Observed counts |
|---|---|---|
| `[10, 80]` | `1, 0, 0, 1`|`1, 0, 0, 1` |
| `[35, 35]` |`0, 0, 2, 0` | `0, 0, 2, 0`|
| `[]` | `0, 0, 0, 0`| `0, 0, 0, 0`|

Stretch: add a counter for all readings. Check that it agrees with the sum of the four category counts. Would this check alone detect a program that incorrectly calls every reading NOMINAL? Explain.

## Class checkpoint

### Save your starting point for the home work
- Repair all marked Part A cells and record the reasoning for each repair.
- Restart the kernel and run Part A in order. Unfinished Part B placeholders may run without errors; that does not mean they are complete.
- Save the notebook, inspect `git status`, and commit your class work separately with a message such as `Complete Lab 03 class work`.
- If class time runs short, record the agreed stopping point here: ______.

The home assignment below is separate. Do not silently add unfinished class activities to it; agree on any catch-up work in class.

## Part B — at home
### B1. From one reading to a sequence

Extend the Lab 02 health monitor. The rules are repeated here so that you can work from a complete specification.

For each independent reading, select the first applicable status:
1. CRITICAL if battery is below 15% **or** voltage is below 22 V.
2. SAFE MODE if battery is below 25%, voltage is below 24 V, temperature is outside the inclusive interval −20°C to 85°C, **or** communications are unavailable.
3. WARNING if battery is at most 35% **or** temperature is above 75°C.
4. NOMINAL otherwise.

All inputs in this task are valid numbers and Boolean values. You do not need to handle missing values, text in numeric fields, or invalid physical values. These records are observations; do not subtract a simulated battery loss from them.

### A small bridge: several values in one record

Each tuple below groups five fields in a fixed order:
`(label, battery_percent, voltage_v, temperature_c, comms_ok)`.

The list holds the records. The five names after `for` receive the five fields of the current record; this is called unpacking. This example is complete, so run it once to see the structure.

In [8]:
example_records = [("Example", 80, 28, 20, True)]

for label, battery_charge, bus_voltage_v, panel_temperature_c, comms_ok in example_records:
    print(label, battery_charge, bus_voltage_v, panel_temperature_c, comms_ok)

Example 80 28 20 True


### Plan before coding

Complete the expected status column from the written rules. Explain D, E, and F, where boundaries and priority interact.

| Label | Battery % | Voltage V | Temperature °C | Comms | Expected status |
|---|---:|---:|---:|---|---|
| A | 80 | 28 | 20 | True | NOMINAL|
| B | 10 | 28 | 20 | True | CRITICAL|
| C | 80 | 21 | 20 | True | CRITICAL|
| D | 15 | 24 | −20 | True | NOMINAL|
| E | 25 | 24 | 85 | True | WARNING|
| F | 35 | 28 | 76 | False | SAFE MODE|
| G | 36 | 28 | 76 | True | WARNING|
| H | 80 | 28 | 20 | True | NOMINAL|

1. D:   Demonstrates how multiple thresholds interact; although battery is 15 and temperature is at the lower boundary, the battery condition $<25$ catches it under SAFE MODE. <br>
2. E:  Shows boundary behavior where battery is exactly 25 and temperature is at the upper boundary (85), falling into WARNING because battery is $\le 35$.<br>
3. F: Illustrates rule priority—even though battery ($\le 35$) and temperature ($> 75$) would qualify for Warning, unavailable communications (False) takes higher priority as a SAFE MODE trigger.<br>

Why is H useful even though it repeats the measurements in A?: It serves as a regression check or bookend to ensure that system state hasn't drifted or corrupted counters after processing a batch of faulty/edge-case readings.

### Implement one report

Use one loop to classify every record, print its label and status, and count each status. After the loop, print four category counts and the total number of records processed. Use the same counter pattern as A5.

A1–A5 used simpler rules for individual measurements. Here you must implement the complete four-input rules above. Reuse your Lab 02 decision chain if useful, then verify it against this specification.

For an empty list, print zero for every summary count. Repeated records count separately. Reset all counters each time the cell runs. Do not add functions, packages, or a user interface; concentrate on the loop and logic.

In [9]:
records = [
    ("A", 80, 28, 20, True),
    ("B", 10, 28, 20, True),
    ("C", 80, 21, 20, True),
    ("D", 15, 24, -20, True),
    ("E", 25, 24, 85, True),
    ("F", 35, 28, 76, False),
    ("G", 36, 28, 76, True),
    ("H", 80, 28, 20, True),
]

# Initialize counters here.
critical_count = 0
safe_count = 0
warning_count = 0
nominal_count = 0
total_processed = 0

for label, battery_charge, bus_voltage_v, panel_temperature_c, comms_ok in records:
    # Full priority decision chain
    if battery_charge < 15 or bus_voltage_v < 22:
        status = "CRITICAL"
        critical_count += 1
    elif battery_charge < 25 or bus_voltage_v < 24 or not (-20 <= panel_temperature_c <= 85) or not comms_ok:
        status = "SAFE MODE"
        safe_count += 1
    elif battery_charge <= 35 or panel_temperature_c > 75:
        status = "WARNING"
        warning_count += 1
    else:
        status = "NOMINAL"
        nominal_count += 1
        
    total_processed += 1
    print(label, status)

# Print the final counts here.
print("\n--- Summary Report ---")
print("CRITICAL:", critical_count)
print("SAFE MODE:", safe_count)
print("WARNING:", warning_count)
print("NOMINAL:", nominal_count)
print("Total records processed:", total_processed)

A NOMINAL
B CRITICAL
C CRITICAL
D SAFE MODE
E WARNING
F SAFE MODE
G WARNING
H NOMINAL

--- Summary Report ---
CRITICAL: 2
SAFE MODE: 2
NOMINAL: 2
Total records processed: 8


### B2. Tests that can expose mistakes

Test the same implementation from B1 by temporarily replacing its `records` list. Do not write a separate classifier for each test. Record predictions before running and restore records A–H afterward.

Required tests:
1. The supplied A–H dataset: compare every status and all category counts with your predictions.
2. `[]`: no per-record output, all summary counts zero.
3. A critical reading followed by a nominal reading, then the reverse order.
4. Three readings with battery 14%, 15%, and 16%, keeping voltage 28 V, temperature 20°C, and communications available.
5. Three readings with voltage 21 V, 22 V, and 23 V, keeping battery 80%, temperature 20°C, and communications available.
6. One reading with both a critical battery and unavailable communications: priority must still be correct.
7. Your own small test that catches a different mistake. State the mistake it would catch.

These tests are evidence, not a proof of correctness. A count total can be correct even when individual statuses are wrong.

| Test | Exact inputs or labels | Expected statuses and counts | Observed result | Explanation of mismatch, if any |
|---|---|---|---|---|
| A–H | Records A–H|A: Nom, B: Crit, C: Crit, D: Safe, E: Warn, F: Safe, G: Warn, H: Nom. <br><br> Counts: 2 Crit, 2 Safe, 2 Warn, 2 Nom |Matches predictions |None|
| Empty input | `[]` |No per-record lines. <br><br> Counts: All 0 | Matches predictions | None|
| Critical then nominal, and reverse | `[("CriticalFirst", 10, 28, 20, True), ("NominalSecond", 80, 28, 20, True)]` & reverse|Order 1: Crit, Nom.<br><br>Order 2: Nom, Crit.<br><br>Counts: 1 Crit, 1 Nom |Matches predictions |None |
| Battery boundary | `[("Bat14", 14, ...), ("Bat15", 15, ...), ("Bat16", 16, ...)]`|Bat14: Critical, Bat15: Safe Mode, Bat16: Warning |Matches predictions |None |
| Voltage boundary |`[("Volt21", ..., 21, ...), ("Volt22", ..., 22, ...), ("Volt23", ..., 23, ...)]` |Volt21: Critical, Volt22: Safe Mode, Volt23: Nominal |Matches predictions |None |
| Competing faults | `[("Comp", 10, 21, 20, False)]`| Comp: Critical (Priority 1 triggers first)| Matches predictions|None |
| My additional test | [("TempEdge", 80, 28, 85, True)]| TempEdge: Nominal (85°C is inclusive)| Matches predictions| Validates inclusive boundary check (`<= 85`)|

What did my additional test detect that the other tests did not? Nothing?

### B3. The context an LLM cannot guess

A colleague asks an LLM:

> Loop over the battery readings and count the alarms.

It proposes the code below. Treat this as a supplied hypothetical response; you do not need an LLM account.

```python
alarm_count = 0
for battery_charge in battery_readings:
    if battery_charge < 25:
        alarm_count = alarm_count + 1
print(alarm_count)
```

Without more context, “alarm” is ambiguous. Consider two possible requirements:

- Requirement A: count every reading below 25%. <br>
- Requirement B: count the start of each run of consecutive readings below 25%. A low first reading starts a run; a reading of 25% or more ends it.

For `[30, 20, 18, 40, 22]`, predict the result under A and under B. Does the proposed code satisfy A, B, both, or neither? Explain why you need the intended requirement before declaring the code wrong.

Now write a replacement prompt for Requirement A. Include the input format and units, exact threshold, treatment of duplicates, empty input behavior, required output, and two examples with expected counts. Ask for a simple loop and an explanation. Use requirements you can verify, not just “make it correct.”

Then explain how you would change the prompt for Requirement B. Implementing B is optional.

Predicted count under A:  count: `3` (readings 20, 18, and 22 are all below 25%). <br>
Predicted count under B:  count: `2` (the drop starting at 20 is run #1; the drop starting at 22 after recovering to 40 is run #2). <br>

Which requirement the proposed code satisfies, and why: <br>
The proposed code satisfies Requirement A. It uses a simple accumulator (`alarm_count = alarm_count + 1`) that increments independently for every individual reading satisfying `battery_charge < 25`, without tracking state or sequence transitions across iterations.   


My replacement prompt for A:
> Write a Python script using a for loop to process a list of numeric battery charge percentages (battery_readings). Count every individual reading that falls exactly below 25%. The input list may contain include duplicates or be empty.

What I would change for B:
> I would instruct the LLM to track state across iterations using a Boolean flag (such as in_alarm_run). The prompt would specify that an alarm count should only increase when transitioning from a normal state ($\ge 25\%$) to a low state ($< 25\%$), and that subsequent consecutive low readings must be ignored until the battery recovers.

Two tests I would use to check the response, and why:
> 1. Test with continuous low readings (`[10, 10, 10]`): This verifies whether the code correctly counts it as only `1` event (Requirement B) rather than `3` separate alarms.
> 2. Test with oscillating states (`[30, 20, 30, 20]`): This checks if the code properly resets its state when the battery recovers, ensuring it correctly counts both distinct drops as `2` separate alarm runs.

```py
battery_readings = [30, 20, 18, 40, 22]

alarm_run_count = 0
previous_was_low = False

for battery_charge in battery_readings:
    if battery_charge < 25:
        if not previous_was_low:
            alarm_run_count = alarm_run_count + 1
            previous_was_low = True
    else:
        previous_was_low = False

print("Number of alarm runs:", alarm_run_count)
```

### B4. Verification record and submission

If you used an LLM, record one useful interaction. If you did not, say so and use the supplied response in B3 for your audit. You do not need to invent an interaction.

| Question | Your record |
|---|---|
| What context did you provide, or what was missing from the supplied prompt? | /|
| What suggestion did you accept, modify, or reject? | /|
| Which exact input and expected output checked it? | /|
| What uncertainty remained after that test? | /|

In 2–4 sentences each:
1. Why do counters belong outside a loop's initialization while each reading needs its own status assignment?
   > Counters must be initialized outside the loop so they can accumulate and remember data across multiple iterations without resetting. However, each reading needs its own status assignment initialized inside the loop so that previous states do not accidentally carry forward into newer and independent readings.
2. Describe a case where the code ran successfully but violated the specification.
   > In `Part A4`, using multiple independent if statements instead of an `if/elif/else` chain caused the code to run completely without syntax or runtime errors, but produced wrong and overlapping classifications
3. Why can a precise prompt improve an LLM answer without making testing unnecessary?
   > A precise prompt reduces ambiguity and guides the AI closer to the intended logic, structure, and constraints.

### Final checklist
- [x] My B1 report follows the complete rules and reports every reading once.
- [x] My tests include boundaries, an empty list, competing faults, and order changes.
- [x] I recorded expected results independently of the implementation.
- [x] I completed the prompt audit and verification record.
- [x] I restored each main example's original input after testing.
- [x] I restarted the kernel and ran all required cells in order, checking their outputs.
- [x] I saved the notebook and committed the home work separately from the class checkpoint.

Use the course's existing submission procedure. Be ready to explain one repair and trace a short sequence without an LLM.

## Optional extension — consecutive events

Implement Requirement B from B3 using a Boolean variable such as `previous_was_low`. Decide what it should mean before the first reading. Update it on every iteration, including normal readings.

Test `[]`, `[20, 18]`, `[20, 30, 18]`, and `[25, 20, 25, 20]`. Predict each event count before running.

Explain why this task intentionally remembers the preceding reading, while the independent status classifier must not reuse the preceding status.

In [10]:
# Optional: count starts of low-battery runs and test your implementation.
test_cases = [
    [],
    [20, 18],
    [20, 30, 18],
    [25, 20, 25, 20]
]

for battery_readings in test_cases:
    alarm_run_count = 0
    previous_was_low = False

    for battery_charge in battery_readings:
        if battery_charge < 25:
            if not previous_was_low:
                alarm_run_count = alarm_run_count + 1
                previous_was_low = True
        else:
            previous_was_low = False

    print(f"Input: {battery_readings} --> Low-battery runs: {alarm_run_count}")

Input: [] --> Low-battery runs: 0
Input: [20, 18] --> Low-battery runs: 1
Input: [20, 30, 18] --> Low-battery runs: 2
Input: [25, 20, 25, 20] --> Low-battery runs: 2
